In [29]:
"""
Multivariate Stats Final - Linear Algebra Concepts in Action
============================================================
從 eigenvector 到 PSD 到 ellipsoid，用 Iris 資料跑一遍 PCA 流程。
每個 section 對應一個概念，跑完你就知道這些概念怎麼用。
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from scipy.stats import chi2

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)


# ============================================================
# SECTION 1: 載入 Iris，看一下資料長什麼樣
# ============================================================
def section1_load_data():
    print("=" * 60)
    print("SECTION 1: 載入 Iris dataset")
    print("=" * 60)
    iris = load_iris()
    X = iris.data        # shape (150, 4): sepal_length, sepal_width, petal_length, petal_width
    y = iris.target      # shape (150,): 0=setosa, 1=versicolor, 2=virginica
    names = iris.feature_names
    
    print(f"資料 shape: {X.shape}")
    print(f"變數名稱: {names}")
    print(f"前 3 筆資料:\n{X[:3]}")
    print(f"\n各變數的 mean: {X.mean(axis=0)}")
    print(f"各變數的 std:  {X.std(axis=0, ddof=1)}")
    
    # Center 資料 (這在 multivariate stats 是標準動作 — 讓 mean = 0)
    mu = X.mean(axis=0)
    Xc = X - mu
    print(f"\nCenter 後的 mean (應該全為 0): {Xc.mean(axis=0)}")
    
    return X, Xc, y, names, mu


In [30]:


# ============================================================
# SECTION 2: Covariance matrix + trace, determinant
#  → Slide 22 的概念
# ============================================================
def section2_covariance(Xc):
    print("\n" + "=" * 60)
    print("SECTION 2: Covariance matrix, trace, determinant")
    print("=" * 60)
    n, p = Xc.shape
    
    # 手算 covariance: Σ = (1/(n-1)) X^T X  (因為已 center)
    Sigma = (Xc.T @ Xc) / (n - 1)
    
    # 用 numpy 驗證
    Sigma_np = np.cov(Xc, rowvar=False)
    assert np.allclose(Sigma, Sigma_np), "手算跟 np.cov 不一致"
    
    print(f"Covariance matrix Σ (4x4):\n{Sigma}")
    print(f"\n是否對稱? {np.allclose(Sigma, Sigma.T)}")  # 必為 True
    
    # Trace 與 determinant
    tr = np.trace(Sigma)
    det = np.linalg.det(Sigma)
    print(f"\ntrace(Σ) = {tr:.4f}  ← total variance (各變數 variance 加起來)")
    print(f"det(Σ)   = {det:.4f}  ← generalized variance (資料雲的『體積』)")
    
    # 驗證 trace = sum of diagonal
    print(f"對角元素之和: {np.diag(Sigma).sum():.4f}  (應等於 trace)")
    
    return Sigma


In [31]:


# ============================================================
# SECTION 3: Eigenvalues + 驗證 Σ 是 PSD
#  → Slide 24
# ============================================================
def section3_eigenvalues_psd(Sigma):
    print("\n" + "=" * 60)
    print("SECTION 3: Eigenvalues + PSD 驗證")
    print("=" * 60)
    # 對稱矩陣用 eigh (比 eig 更穩定，保證實數)
    eigvals, eigvecs = np.linalg.eigh(Sigma)
    # eigh 預設由小到大，反過來方便看
    eigvals = eigvals[::-1]
    eigvecs = eigvecs[:, ::-1]
    
    print(f"Eigenvalues (由大到小): {eigvals}")
    print(f"全部 ≥ 0? {np.all(eigvals >= 0)}  ← 是的話 Σ 就是 PSD")
    print(f"全部 > 0? {np.all(eigvals > 0)}   ← 是的話 Σ 是 PD，可以求逆")
    
    # 驗證 trace 公式: trace = sum of eigenvalues
    print(f"\nSum of eigenvalues: {eigvals.sum():.4f}")
    print(f"trace(Σ) (檢查):    {np.trace(Sigma):.4f}  ← 應該一樣 (slide 22)")
    
    # 驗證 det 公式: det = product of eigenvalues
    print(f"\nProduct of eigenvalues: {np.prod(eigvals):.6f}")
    print(f"det(Σ) (檢查):          {np.linalg.det(Sigma):.6f}  ← 應該一樣")
    
    return eigvals, eigvecs


In [32]:


# ============================================================
# SECTION 4: 驗證 A v = λ v (eigenvector 的定義)
#  → Slide 14
# ============================================================
def section4_verify_eigenvectors(Sigma, eigvals, eigvecs):
    print("\n" + "=" * 60)
    print("SECTION 4: 驗證 Σv = λv (eigenvector 的定義)")
    print("=" * 60)
    for i in range(len(eigvals)):
        v = eigvecs[:, i]
        Av = Sigma @ v
        lam_v = eigvals[i] * v
        print(f"\nEigenvector v_{i+1} = {v}")
        print(f"  Σv     = {Av}")
        print(f"  λ_{i+1} v = {lam_v}")
        print(f"  相等? {np.allclose(Av, lam_v)}")
        print(f"  ||v|| = {np.linalg.norm(v):.4f}  ← 應為 1 (unit length)")
    
    # 驗證 eigenvectors 兩兩正交
    print("\n--- 驗證 eigenvectors 互相正交 (v_i · v_j = 0 for i≠j) ---")
    print(f"Q^T Q =\n{eigvecs.T @ eigvecs}  ← 應為 I (對角為 1，其他為 0)")


In [33]:


# ============================================================
# SECTION 5: Spectral decomposition Σ = Q D Q^T
#  → Slide 20
# ============================================================
def section5_spectral_decomposition(Sigma, eigvals, eigvecs):
    print("\n" + "=" * 60)
    print("SECTION 5: Spectral decomposition Σ = Q D Q^T")
    print("=" * 60)
    Q = eigvecs
    D = np.diag(eigvals)
    
    Sigma_reconstructed = Q @ D @ Q.T
    print(f"原始 Σ:\n{Sigma}")
    print(f"\nQ D Q^T 重建:\n{Sigma_reconstructed}")
    print(f"\n相等? {np.allclose(Sigma, Sigma_reconstructed)}")
    
    # 也驗證 Q 是 orthogonal: Q Q^T = I
    print(f"\nQ Q^T (也應為 I):\n{Q @ Q.T}")
    
    return Q, D



In [34]:

# ============================================================
# SECTION 6: PCA 投影 + 畫橢球
#  → Slide 26 + 27
# ============================================================
def section6_pca_and_ellipse(Xc, y, eigvals, eigvecs, names):
    print("\n" + "=" * 60)
    print("SECTION 6: PCA 投影 + 信賴橢圓")
    print("=" * 60)
    
    # PCA = 把 centered data 投影到 eigenvectors 上
    # Y = Xc @ Q  → Y 的每個 column 是資料在某個 eigenvector 方向上的座標
    Y = Xc @ eigvecs
    
    # 驗證: Y 的 covariance 應該是對角矩陣 D，對角元素 = eigenvalues
    Y_cov = np.cov(Y, rowvar=False)
    print(f"投影後資料的 covariance (應為對角矩陣，對角 = eigenvalues):\n{Y_cov}")
    print(f"\n各方向的 variance: {np.diag(Y_cov)}")
    print(f"Eigenvalues:        {eigvals}")
    print("→ 完全一樣! 這就是『eigenvalue = 該方向的 variance』")
    
    # Variance explained ratio
    total_var = eigvals.sum()
    print(f"\n各 PC 解釋多少 variance:")
    for i, lam in enumerate(eigvals):
        print(f"  PC{i+1}: {lam/total_var*100:.2f}%  (累積: {eigvals[:i+1].sum()/total_var*100:.2f}%)")
    print("→ 前兩個 PC 已經涵蓋 ~98%，所以降到 2D 幾乎沒丟資訊")
    
    # 畫圖: 在 PC1-PC2 平面上看 3 個物種，並畫 95% 信賴橢圓
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 左: 原始空間中前兩個變數
    colors = ['#e41a1c', '#377eb8', '#4daf4a']
    species = ['setosa', 'versicolor', 'virginica']
    ax = axes[0]
    for k in range(3):
        mask = (y == k)
        ax.scatter(Xc[mask, 0], Xc[mask, 1], c=colors[k], label=species[k], alpha=0.7, s=40)
    ax.set_xlabel(names[0] + ' (centered)')
    ax.set_ylabel(names[1] + ' (centered)')
    ax.set_title('Original space: sepal length vs sepal width\n(classes overlap a lot)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    
    # 右: PC1-PC2 平面
    ax = axes[1]
    for k in range(3):
        mask = (y == k)
        ax.scatter(Y[mask, 0], Y[mask, 1], c=colors[k], label=species[k], alpha=0.7, s=40)
    
    # 畫 95% 橢圓: x^T Σ_2^(-1) x = chi²(2, 0.95)
    c = chi2.ppf(0.95, df=2)  # ≈ 5.99
    theta = np.linspace(0, 2*np.pi, 200)
    a, b = np.sqrt(c * eigvals[0]), np.sqrt(c * eigvals[1])
    ex = a * np.cos(theta)
    ey = b * np.sin(theta)
    ax.plot(ex, ey, 'k--', lw=2, label='95% MVN contour')
    
    # 標記半軸
    ax.plot([0, a], [0, 0], 'k-', lw=2)
    ax.plot([0, 0], [0, b], 'k-', lw=2)
    ax.annotate(f'semi-axis 1 = sqrt(c*lam_1) = {a:.2f}', xy=(a/2 - 1, 0.15), fontsize=9)
    ax.annotate(f'semi-axis 2\n= sqrt(c*lam_2)\n= {b:.2f}', xy=(0.1, b/2 - 0.5), fontsize=9)
    
    ax.set_xlabel(f'PC1 (variance = lam_1 = {eigvals[0]:.3f})')
    ax.set_ylabel(f'PC2 (variance = lam_2 = {eigvals[1]:.3f})')
    ax.set_title('PCA plane: ellipse axes = eigenvectors\nsemi-axis length = sqrt(c*lam_i)  (slide 26)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig('/Users/fangsiyu/Desktop/sdu-2026-code/805_multivariate_statistical_analysis/note/fig1_pca_ellipse.png', dpi=110, bbox_inches='tight')
    plt.close()
    print("\n→ 圖存到 fig1_pca_ellipse.png")
    
    return Y


In [35]:


# ============================================================
# SECTION 7: Mahalanobis distance (quadratic form 當距離)
#  → Slide 23, 27
# ============================================================
def section7_mahalanobis(Xc, Sigma):
    print("\n" + "=" * 60)
    print("SECTION 7: Mahalanobis distance (quadratic form)")
    print("=" * 60)
    
    # Mahalanobis distance²: d² = x^T Σ^(-1) x   (x 已 center)
    # 這是個 quadratic form! 跟 slide 23 的 x^T A x 一模一樣
    Sigma_inv = np.linalg.inv(Sigma)
    
    # 對每個點算 d²
    # 寫成 vector form: d² = sum over i,j of x_i * Σ^(-1)_{ij} * x_j
    d_sq = np.einsum('ni,ij,nj->n', Xc, Sigma_inv, Xc)
    
    print(f"前 5 個點的 Mahalanobis distance²: {d_sq[:5]}")
    print(f"全部 d² ≥ 0? {np.all(d_sq >= 0)}")
    print("→ 永遠非負，這就是 Σ^(-1) 是 PD 的保證 (slide 23)")
    
    # 理論: 如果資料是 MVN，d² ~ χ²(p) where p = 變數數 = 4
    # 期望值 = p, variance = 2p
    p = Xc.shape[1]
    print(f"\nMahalanobis d² 的統計性質:")
    print(f"  Mean(d²)   = {d_sq.mean():.4f}  (理論 χ²(p) 的 mean = p = {p})")
    print(f"  Var(d²)    = {d_sq.var():.4f}  (理論 = 2p = {2*p})")
    print("→ 接近但不完全 = 因為 Iris 不是真的 MVN (有 3 個 cluster)")
    
    # 用 d² 找 outliers (與 cluster 中心『距離』最遠的點)
    threshold = chi2.ppf(0.975, df=p)
    outliers = np.where(d_sq > threshold)[0]
    print(f"\n95% 卡方臨界值 (df={p}): {threshold:.4f}")
    print(f"被視為 outlier 的點 index: {outliers}  (共 {len(outliers)} 個)")
    
    # 畫 d² 分布
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(d_sq, bins=30, density=True, alpha=0.6, color='steelblue', edgecolor='black')
    
    # 疊上 χ²(p) 理論曲線
    from scipy.stats import chi2 as chi2_dist
    xs = np.linspace(0, d_sq.max() * 1.1, 200)
    ax.plot(xs, chi2_dist.pdf(xs, df=p), 'r-', lw=2, label=f'chi^2(p={p}) theoretical')
    ax.axvline(threshold, color='black', linestyle='--', label=f'97.5% threshold = {threshold:.2f}')
    ax.set_xlabel('Mahalanobis distance^2 = x^T Sigma^(-1) x')
    ax.set_ylabel('Density')
    ax.set_title('Mahalanobis d^2 distribution vs chi^2(4)\n(under MVN assumption, should match exactly)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/Users/fangsiyu/Desktop/sdu-2026-code/805_multivariate_statistical_analysis/note/fig2_mahalanobis.png', dpi=110, bbox_inches='tight')
    plt.close()
    print("\n→ 圖存到 fig2_mahalanobis.png")
    
    return d_sq


In [36]:


# ============================================================
# SECTION 8: Square root matrix → Whitening
#  → Slide 25
# ============================================================
def section8_whitening(Xc, Sigma, eigvals, eigvecs):
    print("\n" + "=" * 60)
    print("SECTION 8: Square root matrix + Whitening")
    print("=" * 60)
    
    # Σ^(-1/2) = Q D^(-1/2) Q^T   (slide 25)
    Q = eigvecs
    D_inv_sqrt = np.diag(1.0 / np.sqrt(eigvals))
    Sigma_inv_sqrt = Q @ D_inv_sqrt @ Q.T
    
    # 驗證: Σ^(-1/2) Σ Σ^(-1/2) = I
    check = Sigma_inv_sqrt @ Sigma @ Sigma_inv_sqrt
    print(f"Σ^(-1/2) Σ Σ^(-1/2) = (應為 I):\n{check}")
    print(f"接近 I? {np.allclose(check, np.eye(4))}")
    
    # Whitening: Z = Σ^(-1/2) Xc
    # Z 的 covariance 會是 I (球形分布)
    Z = Xc @ Sigma_inv_sqrt.T
    Z_cov = np.cov(Z, rowvar=False)
    print(f"\nWhitened data 的 covariance (應為 I):\n{Z_cov}")
    
    # 驗證: Mahalanobis distance² 在 whitened space = 普通 Euclidean 距離平方
    d_mahalanobis_sq = np.einsum('ni,ij,nj->n', Xc, np.linalg.inv(Sigma), Xc)
    d_euclidean_sq_whitened = np.sum(Z**2, axis=1)
    print(f"\n前 5 個點:")
    print(f"  Mahalanobis d² (原空間):  {d_mahalanobis_sq[:5]}")
    print(f"  Euclidean d² (whitened): {d_euclidean_sq_whitened[:5]}")
    print(f"  相等? {np.allclose(d_mahalanobis_sq, d_euclidean_sq_whitened)}")
    print("→ Whitening 把『有相關性的橢球資料』轉成『獨立的球形資料』")
    print("→ 這就是『square root matrix 把 quadratic form 變成 sum of squares』(slide 25)")
    
    # 視覺化 before / after
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    
    # Before: 原始 centered data (前兩個變數)
    ax = axes[0]
    ax.scatter(Xc[:, 0], Xc[:, 1], alpha=0.6, s=30)
    ax.set_xlabel('sepal length (centered)')
    ax.set_ylabel('sepal width (centered)')
    ax.set_title('Before: original centered data\n(elliptical shape, variables correlated)')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_aspect('equal')
    
    # After: whitened data (前兩個維度)
    ax = axes[1]
    ax.scatter(Z[:, 0], Z[:, 1], alpha=0.6, s=30, color='darkorange')
    ax.set_xlabel('whitened dim 1')
    ax.set_ylabel('whitened dim 2')
    ax.set_title('After: whitened data Z = Sigma^(-1/2) X_c\n(spherical shape, covariance = I)')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig('/Users/fangsiyu/Desktop/sdu-2026-code/805_multivariate_statistical_analysis/note/fig3_whitening.png', dpi=110, bbox_inches='tight')
    plt.close()
    print("\n→ 圖存到 fig3_whitening.png")



In [37]:

# ============================================================
# Run everything
# ============================================================
if __name__ == "__main__":
    from scipy.stats import chi2
    X, Xc, y, names, mu = section1_load_data()
    Sigma = section2_covariance(Xc)
    eigvals, eigvecs = section3_eigenvalues_psd(Sigma)
    section4_verify_eigenvectors(Sigma, eigvals, eigvecs)
    Q, D = section5_spectral_decomposition(Sigma, eigvals, eigvecs)
    Y = section6_pca_and_ellipse(Xc, y, eigvals, eigvecs, names)
    d_sq = section7_mahalanobis(Xc, Sigma)
    section8_whitening(Xc, Sigma, eigvals, eigvecs)
    
    print("\n" + "=" * 60)
    print("全部跑完!")
    print("=" * 60)

SECTION 1: 載入 Iris dataset
資料 shape: (150, 4)
變數名稱: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
前 3 筆資料:
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]]

各變數的 mean: [5.8433 3.0573 3.758  1.1993]
各變數的 std:  [0.8281 0.4359 1.7653 0.7622]

Center 後的 mean (應該全為 0): [-0. -0. -0. -0.]

SECTION 2: Covariance matrix, trace, determinant
Covariance matrix Σ (4x4):
[[ 0.6857 -0.0424  1.2743  0.5163]
 [-0.0424  0.19   -0.3297 -0.1216]
 [ 1.2743 -0.3297  3.1163  1.2956]
 [ 0.5163 -0.1216  1.2956  0.581 ]]

是否對稱? True

trace(Σ) = 4.5730  ← total variance (各變數 variance 加起來)
det(Σ)   = 0.0019  ← generalized variance (資料雲的『體積』)
對角元素之和: 4.5730  (應等於 trace)

SECTION 3: Eigenvalues + PSD 驗證
Eigenvalues (由大到小): [4.2282 0.2427 0.0782 0.0238]
全部 ≥ 0? True  ← 是的話 Σ 就是 PSD
全部 > 0? True   ← 是的話 Σ 是 PD，可以求逆

Sum of eigenvalues: 4.5730
trace(Σ) (檢查):    4.5730  ← 應該一樣 (slide 22)

Product of eigenvalues: 0.001913
det(Σ) (檢查):          0.001913  ← 應該一樣

SECTION 4: 驗證